# Compare equivalent custom masking policies

Use this folder/shared kernel. The exact same five strings go through both native frameworks. Inputs and separate sensitivity expectations are synthetic AI-authored drafts, not human judgments. The public documentation placeholder is intentionally expected to remain visible, exposing a context-free matcher trade-off.

In [ ]:
import os
os.environ.update(DO_NOT_TRACK="1", OTEL_SDK_DISABLED="true", HF_HUB_OFFLINE="1", LITELLM_LOCAL_MODEL_COST_MAP="true")
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/strings.json").read_text())
expectations = json.loads(Path("docs/goldends/draft_expectations.json").read_text())
pprint(fixtures)

## Read the native framework helper and configuration

`tools/compare.py` sends one shared regex to actual NeMo registered output actions and a registered Guardrails AI FIX validator. No pretrained detector/model is involved. Constructors run outside each timed check; timing helpers measure only actual check_async/parse calls. Read config/rails.co and the helper first.

In [ ]:
from tools.compare import email_pattern, build_nemo, build_guardrails, nemo_check, guardrails_check
pattern = email_pattern(include_obfuscated=False)
baseline_events = []
nemo = build_nemo(pattern, baseline_events)
guardrails = build_guardrails()
pprint({"identical_pattern_for_both": pattern, "timed_setup": False})

## Baseline on identical strings

Inspect actual SDK status, before/after text, registered-action evidence, native validator history, policy invocation count, and measured local duration. Extra guard/answer-model calls and re-asks are zero.

The paired loop awaits NeMo, then calls synchronous Guard.parse. Guardrails AI emits its actual event-loop warning and falls back to synchronous validation; this is not an async Guardrails check. Ordered calls and cold/first-call effects make timings noisy and unsuitable for a speed ranking.


In [ ]:
baseline = {}
for item in fixtures:
    text = item["text"]
    baseline[item["id"]] = {"nemo": await nemo_check(nemo, text, baseline_events), "guardrails": guardrails_check(guardrails, text, pattern)}
for name, results in baseline.items():
    pprint({"id": name, **results})

## A repaired final pass still had an original failure

Guardrails AI FIX passes after masking the detected email, while its validator history retains the original failure. Count that as a detected original pattern. NeMo reports modified and records a matched output action for the same string.

In [ ]:
pprint(baseline["conventional"])
assert baseline["conventional"]["guardrails"]["native_validation_passed"] is True
assert baseline["conventional"]["guardrails"]["detected_original_pattern"] is True
assert baseline["conventional"]["nemo"]["native_status"] == "modified"
assert all(results["nemo"]["exposed"] == results["guardrails"]["exposed"] for results in baseline.values())

## Change only include_obfuscated for both frameworks

Use one new pattern value for both, keeping the exact same input strings, replacement text, and FIX behavior. Setup is again outside each timed check. The candidate additionally matches the simple at/dot spelling.

In [ ]:
candidate_pattern = email_pattern(include_obfuscated=True)
candidate_events = []
candidate_nemo = build_nemo(candidate_pattern, candidate_events)
candidate_guardrails = build_guardrails()
candidate = {}
for item in fixtures:
    text = item["text"]
    candidate[item["id"]] = {"nemo": await nemo_check(candidate_nemo, text, candidate_events), "guardrails": guardrails_check(candidate_guardrails, text, candidate_pattern)}
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "baseline": baseline[name]["nemo"]["exposed"], "candidate_nemo": candidate[name]["nemo"]["exposed"], "candidate_guardrails": candidate[name]["guardrails"]["exposed"]})

## Compare draft sensitivity errors and measured local work

Both policies catch one more draft-sensitive contact but still miss bracketed spelling and mask the public placeholder. Single tiny timings are noisy; identical transforms here do not establish universal framework superiority, comprehensive PII coverage, or truth.

In [ ]:
for setting, results in [("baseline", baseline), ("candidate", candidate)]:
    for framework in ["nemo", "guardrails"]:
        rows = {name: result[framework] for name, result in results.items()}
        caught = sum(expectations["should_mask"][name] and row["detected_original_pattern"] for name, row in rows.items())
        missed = sum(expectations["should_mask"][name] and not row["detected_original_pattern"] for name, row in rows.items())
        accidental = sum(not expectations["should_mask"][name] and row["detected_original_pattern"] for name, row in rows.items())
        pprint({"setting": setting, "framework": framework, "caught_vs_AI_draft": caught, "missed_vs_AI_draft": missed, "accidental_masks_vs_AI_draft": accidental,
                "actual_policy_invocations": sum(row["actual_policy_invocations"] for row in rows.values()), "measured_total_ms": sum(row["measured_ms"] for row in rows.values()), "guard_model_calls": 0, "answer_model_calls": 0, "reask_calls": 0})
assert all(results["nemo"]["exposed"] == results["guardrails"]["exposed"] for results in candidate.values())
assert candidate["bracketed"]["nemo"]["detected_original_pattern"] is False
assert candidate["public_example"]["guardrails"]["detected_original_pattern"] is True